In [1]:
!pip install flask groq PyPDF2 pyngrok -q
print('Libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.3 MB/s eta 0:00:00
Libraries installed!


In [2]:
import os
os.makedirs('/content/resume_analyzer/templates', exist_ok=True)
os.chdir('/content/resume_analyzer')
print('Folder ready')

Folder ready


In [3]:
app_code = r'''
import os, json, re, io
from flask import Flask, render_template, request, jsonify
from groq import Groq
import PyPDF2

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024
client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

def read_pdf(file):
    reader = PyPDF2.PdfReader(file)
    return "".join(p.extract_text() + "\n" for p in reader.pages).strip()

def get_label(score):
    if score >= 75: return "Excellent"
    if score >= 45: return "Good Start"
    return "Needs Work"

def make_prompt(text, role):
    return f"""You are a strict HR recruiter and ATS expert reviewing a resume for: "{role}"
Rules: be specific to THIS resume, mention real candidate details, no generic advice.

RESUME:
---
{text}
---

Return ONLY valid JSON, no markdown, no extra text:
{{
  "overall_score": <0-100>,
  "ats_score": <0-100>,
  "sections": {{"tone_style": <0-100>, "content": <0-100>, "structure": <0-100>, "skills": <0-100>}},
  "strong_points": ["<specific point from resume>", "<specific point>", "<specific point>"],
  "weak_points": ["<specific gap in this resume>", "<specific gap>", "<specific gap>"],
  "missing_keywords": ["<keyword for {role} not in resume>", "<keyword>", "<keyword>", "<keyword>", "<keyword>"],
  "recommendations": [
    {{"priority": "High",   "action": "<specific fix referencing actual resume content>"}},
    {{"priority": "High",   "action": "<specific fix referencing actual resume content>"}},
    {{"priority": "Medium", "action": "<specific improvement for this resume>"}},
    {{"priority": "Medium", "action": "<specific improvement for this resume>"}},
    {{"priority": "Low",    "action": "<small polish item>"}}
  ],
  "summary": "<3 sentences: candidate name, fit for {role}, top thing to fix>"
}}"""

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        if "resume" not in request.files:
            return jsonify({"error": "No file uploaded"}), 400
        f = request.files["resume"]
        role = request.form.get("job_role", "").strip()
        if not f.filename or not role:
            return jsonify({"error": "Missing file or job role"}), 400
        if not f.filename.endswith(".pdf"):
            return jsonify({"error": "Please upload a PDF file"}), 400

        text = read_pdf(io.BytesIO(f.read()))
        if len(text) < 50:
            return jsonify({"error": "Cannot read PDF text — is it a scanned image?"}), 400

        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Return valid JSON only. No markdown. Be specific to the resume."},
                {"role": "user",   "content": make_prompt(text, role)}
            ],
            temperature=0.2,
            max_tokens=2000
        )

        data = json.loads(re.sub(r"```json|```", "", resp.choices[0].message.content).strip())
        s = data.get("sections", {})
        data["section_labels"] = {k: get_label(s.get(k, 0)) for k in ["tone_style","content","structure","skills"]}
        return jsonify({"success": True, "job_role": role, "analysis": data})

    except json.JSONDecodeError:
        return jsonify({"error": "AI gave bad response, please try again"}), 500
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(port=5000)
'''

with open('/content/resume_analyzer/app.py', 'w') as f:
    f.write(app_code)
print('app.py created!')

app.py created!


In [6]:
html = """<!DOCTYPE html>
<html lang=\"en\">
<head>
  <meta charset=\"UTF-8\"/>
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\"/>
  <title>ResumeIQ</title>
  <link href=\"https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap\" rel=\"stylesheet\"/>
  <style>
    :root{--bg:#0a0a0f;--surface:#13131a;--card:#1a1a24;--border:#2a2a38;--accent:#7c5cfc;--accent2:#fc5c7c;--text:#e8e8f0;--muted:#7878a0;--good:#5cfcb4;--warn:#fcb85c;--bad:#fc5c7c}
    *{box-sizing:border-box;margin:0;padding:0}
    body{background:var(--bg);color:var(--text);font-family:'DM Sans',sans-serif;min-height:100vh}
    body::before{content:'';position:fixed;inset:0;background-image:linear-gradient(rgba(124,92,252,0.04) 1px,transparent 1px),linear-gradient(90deg,rgba(124,92,252,0.04) 1px,transparent 1px);background-size:60px 60px;pointer-events:none;z-index:0}
    .container{max-width:1100px;margin:0 auto;padding:0 24px;position:relative;z-index:1}
    header{padding:28px 0 24px;border-bottom:1px solid var(--border);margin-bottom:48px}
    .logo{font-family:'Syne',sans-serif;font-size:1.5rem;font-weight:800;background:linear-gradient(135deg,var(--accent),var(--accent2));-webkit-background-clip:text;-webkit-text-fill-color:transparent}
    .hero{text-align:center;margin-bottom:56px}
    .hero h1{font-family:'Syne',sans-serif;font-size:clamp(2.2rem,5vw,3.5rem);font-weight:800;line-height:1.1;letter-spacing:-1.5px;margin-bottom:16px}
    .hero h1 .hl{background:linear-gradient(135deg,var(--accent),var(--accent2));-webkit-background-clip:text;-webkit-text-fill-color:transparent}
    .hero p{color:var(--muted);font-size:1.1rem;max-width:500px;margin:0 auto}
    .upload-card{background:var(--card);border:1px solid var(--border);border-radius:20px;padding:40px;margin-bottom:40px;position:relative;overflow:hidden}
    .upload-card::before{content:'';position:absolute;top:-60px;right:-60px;width:200px;height:200px;background:radial-gradient(circle,rgba(124,92,252,0.15),transparent 70%);pointer-events:none}
    .form-grid{display:grid;grid-template-columns:1fr 1fr;gap:24px;margin-bottom:28px}
    @media(max-width:640px){.form-grid{grid-template-columns:1fr}}
    .form-group label{display:block;font-size:.8rem;font-weight:500;letter-spacing:.08em;text-transform:uppercase;color:var(--muted);margin-bottom:10px}
    .drop-zone{border:2px dashed var(--border);border-radius:14px;padding:36px 20px;text-align:center;cursor:pointer;transition:all .3s;background:var(--surface);position:relative}
    .drop-zone:hover,.drop-zone.drag{border-color:var(--accent);background:rgba(124,92,252,0.06)}
    .drop-zone input{position:absolute;inset:0;opacity:0;cursor:pointer;width:100%;height:100%}
    .drop-icon{font-size:2.5rem;margin-bottom:10px;display:block}
    .drop-zone p{color:var(--muted);font-size:.9rem}
    .fname{color:var(--accent);font-weight:500;font-size:.95rem;margin-top:8px;display:none}
    .role-input{width:100%;background:var(--surface);border:1.5px solid var(--border);border-radius:14px;padding:16px 20px;color:var(--text);font-family:'DM Sans',sans-serif;font-size:1rem;transition:border-color .3s;outline:none}
    .role-input:focus{border-color:var(--accent)}
    .role-input::placeholder{color:var(--muted)}
    .tags{display:flex;flex-wrap:wrap;gap:8px;margin-top:12px}
    .tag{background:var(--surface);border:1px solid var(--border);border-radius:20px;padding:5px 14px;font-size:.8rem;color:var(--muted);cursor:pointer;transition:all .2s}
    .tag:hover{border-color:var(--accent);color:var(--accent)}
    .btn{width:100%;padding:18px;background:linear-gradient(135deg,var(--accent),var(--accent2));border:none;border-radius:14px;color:#fff;font-family:'Syne',sans-serif;font-size:1.05rem;font-weight:700;cursor:pointer;transition:all .3s}
    .btn:hover{transform:translateY(-2px);box-shadow:0 12px 40px rgba(124,92,252,0.4)}
    .btn:disabled{opacity:.6;cursor:not-allowed;transform:none}
    .loading{display:none;text-align:center;padding:60px 20px}
    .spinner{width:56px;height:56px;border:3px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:spin .9s linear infinite;margin:0 auto 20px}
    @keyframes spin{to{transform:rotate(360deg)}}
    .loading p{color:var(--muted);font-size:.95rem}
    .score-hero{display:grid;grid-template-columns:auto 1fr;gap:32px;align-items:center;background:var(--card);border:1px solid var(--border);border-radius:20px;padding:36px 40px;margin-bottom:24px}
    @media(max-width:640px){.score-hero{grid-template-columns:1fr;text-align:center}}
    .big-score{width:120px;height:120px;display:flex;flex-direction:column;align-items:center;justify-content:center;position:relative;flex-shrink:0}
    .big-score svg{position:absolute;top:0;left:0;transform:rotate(-90deg)}
    .big-score .num{font-family:'Syne',sans-serif;font-size:2rem;font-weight:800;line-height:1;position:relative;z-index:1}
    .big-score .lbl{font-size:.65rem;color:var(--muted);letter-spacing:.08em;text-transform:uppercase;position:relative;z-index:1}
    .score-info h2{font-family:'Syne',sans-serif;font-size:1.5rem;font-weight:700;margin-bottom:8px}
    .score-info .sum{color:var(--muted);line-height:1.6;font-size:.95rem}
    .sgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:16px;margin-bottom:24px}
    @media(max-width:640px){.sgrid{grid-template-columns:1fr}}
    .sc{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:22px 24px}
    .sch{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px}
    .sct{font-weight:600;font-size:.95rem}
    .badge{padding:3px 10px;border-radius:20px;font-size:.72rem;font-weight:600;letter-spacing:.05em}
    .bg{background:rgba(92,252,180,0.12);color:var(--good)}
    .bw{background:rgba(252,184,92,0.12);color:var(--warn)}
    .bb{background:rgba(252,92,124,0.12);color:var(--bad)}
    .track{background:var(--surface);border-radius:99px;height:8px;overflow:hidden;margin-bottom:8px}
    .fill{height:100%;border-radius:99px;transition:width 1.2s cubic-bezier(0.4,0,0.2,1);width:0}
    .snr{display:flex;justify-content:space-between;font-size:.8rem;color:var(--muted)}
    .snr strong{color:var(--text);font-size:.9rem}
    .ats-card{background:rgba(252,92,124,0.05);border:1px solid rgba(252,92,124,0.2);border-radius:16px;padding:24px;margin-bottom:24px}
    .ats-h{display:flex;align-items:center;gap:12px;margin-bottom:16px}
    .ats-h h3{font-family:'Syne',sans-serif;font-weight:700;font-size:1.1rem}
    .ats-big{font-family:'Syne',sans-serif;font-size:2rem;font-weight:800;color:var(--accent2)}
    .ats-list{list-style:none}
    .ats-list li{padding:8px 0;border-bottom:1px solid rgba(255,255,255,0.05);display:flex;align-items:flex-start;gap:10px;font-size:.9rem;color:var(--muted)}
    .ats-list li:last-child{border-bottom:none}
    .ats-list li::before{content:'\u26a0';color:var(--warn);flex-shrink:0;margin-top:1px}
    .two{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:24px}
    @media(max-width:640px){.two{grid-template-columns:1fr}}
    .panel{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:24px}
    .panel h3{font-family:'Syne',sans-serif;font-weight:700;font-size:1rem;margin-bottom:16px}
    .panel ul{list-style:none}
    .panel ul li{padding:8px 0;border-bottom:1px solid rgba(255,255,255,0.04);font-size:.875rem;color:var(--muted);display:flex;align-items:flex-start;gap:10px;line-height:1.5}
    .panel ul li:last-child{border-bottom:none}
    .dot{width:7px;height:7px;border-radius:50%;flex-shrink:0;margin-top:6px}
    .dg{background:var(--good)}.dr{background:var(--bad)}
    .kw-card{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:24px;margin-bottom:24px}
    .kw-card h3{font-family:'Syne',sans-serif;font-weight:700;font-size:1rem;margin-bottom:16px}
    .chips{display:flex;flex-wrap:wrap;gap:8px}
    .chip{background:rgba(252,92,124,0.1);border:1px solid rgba(252,92,124,0.25);color:var(--bad);border-radius:20px;padding:5px 14px;font-size:.8rem;font-weight:500}
    .rec-card{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:24px;margin-bottom:40px}
    .rec-card h3{font-family:'Syne',sans-serif;font-weight:700;font-size:1rem;margin-bottom:20px}
    .rec{display:flex;gap:16px;padding:14px 0;border-bottom:1px solid rgba(255,255,255,0.04)}
    .rec:last-child{border-bottom:none}
    .pri{flex-shrink:0;padding:3px 10px;border-radius:20px;font-size:.72rem;font-weight:700;letter-spacing:.06em;height:fit-content;margin-top:2px}
    .ph{background:rgba(252,92,124,0.15);color:var(--bad)}
    .pm{background:rgba(252,184,92,0.15);color:var(--warn)}
    .pl{background:rgba(92,252,180,0.12);color:var(--good)}
    .rt{font-size:.9rem;color:var(--muted);line-height:1.55}
    .err{display:none;background:rgba(252,92,124,0.08);border:1px solid rgba(252,92,124,0.3);border-radius:14px;padding:18px 24px;color:var(--bad);font-size:.9rem;margin-bottom:20px}
    .again{display:block;margin:0 auto 60px;padding:14px 36px;background:transparent;border:1.5px solid var(--border);border-radius:12px;color:var(--muted);font-family:'DM Sans',sans-serif;font-size:.95rem;cursor:pointer;transition:all .2s}
    .again:hover{border-color:var(--accent);color:var(--accent)}
    .fi{animation:fi 0.5s ease forwards}
    @keyframes fi{from{opacity:0;transform:translateY(16px)}to{opacity:1;transform:translateY(0)}}
  </style>
</head>
<body>
<header><div class=\"container\"><div class=\"logo\">ResumeIQ \u2756</div></div></header>
<div class=\"container\">
  <div class=\"hero\">
    <h1>Analyze Your Resume<br/>with <span class=\"hl\">AI Precision</span></h1>
    <p>Instant ATS scores, skill gap analysis, and personalized recommendations</p>
  </div>
  <div class=\"upload-card\" id=\"upload-section\">
    <div class=\"form-grid\">
      <div class=\"form-group\">
        <label>📄 Upload Resume (PDF)</label>
        <div class=\"drop-zone\" id=\"dz\">
          <input type=\"file\" id=\"rf\" accept=\".pdf\"/>
          <span class=\"drop-icon\">📁</span>
          <p>Drag & drop or <strong>click to browse</strong></p>
          <div class=\"fname\" id=\"fn\"></div>
        </div>
      </div>
      <div class=\"form-group\">
        <label>🎯 Target Job Role</label>
        <input type=\"text\" class=\"role-input\" id=\"jr\" placeholder=\"e.g. Full Stack Developer\"/>
        <div class=\"tags\">
          <div class=\"tag\" onclick=\"setRole('Software Engineer')\">Software Engineer</div>
          <div class=\"tag\" onclick=\"setRole('Data Analyst')\">Data Analyst</div>
          <div class=\"tag\" onclick=\"setRole('Frontend Developer')\">Frontend Dev</div>
          <div class=\"tag\" onclick=\"setRole('ML Engineer')\">ML Engineer</div>
          <div class=\"tag\" onclick=\"setRole('Product Manager')\">Product Manager</div>
        </div>
      </div>
    </div>
    <div class=\"err\" id=\"err\"></div>
    <button class=\"btn\" id=\"abtn\" onclick=\"run()\">\u2756 Analyze My Resume</button>
  </div>
  <div class=\"loading\" id=\"loading\"><div class=\"spinner\"></div><p>AI is reading your resume\u2026 10-20 seconds</p></div>
  <div id=\"results\"></div>
</div>
<script>
const dz=document.getElementById('dz'),rf=document.getElementById('rf'),fn=document.getElementById('fn');
rf.onchange=()=>{ if(rf.files[0]){fn.textContent='\u2713 '+rf.files[0].name;fn.style.display='block'} };
dz.addEventListener('dragover',e=>{e.preventDefault();dz.classList.add('drag')});
dz.addEventListener('dragleave',()=>dz.classList.remove('drag'));
dz.addEventListener('drop',e=>{e.preventDefault();dz.classList.remove('drag');const f=e.dataTransfer.files[0];if(f&&f.type==='application/pdf'){rf.files=e.dataTransfer.files;fn.textContent='\u2713 '+f.name;fn.style.display='block'}});
function setRole(r){document.getElementById('jr').value=r}
async function run(){
  const file=rf.files[0],role=document.getElementById('jr').value.trim(),err=document.getElementById('err');
  err.style.display='none';
  if(!file)return showErr('Please upload a PDF resume.');
  if(!role)return showErr('Please enter a job role.');
  const fd=new FormData();
  fd.append('resume',file);fd.append('job_role',role);
  document.getElementById('loading').style.display='block';
  document.getElementById('abtn').disabled=true;
  document.getElementById('results').style.display='none';
  try{
    const res=await fetch('/analyze',{method:'POST',body:fd});
    const d=await res.json();
    document.getElementById('loading').style.display='none';
    if(d.success) show(d.analysis,d.job_role);
    else{showErr(d.error||'Something went wrong');document.getElementById('abtn').disabled=false}
  }catch(e){
    document.getElementById('loading').style.display='none';
    showErr('Network error.');
    document.getElementById('abtn').disabled=false;
  }
}
function showErr(m){const e=document.getElementById('err');e.textContent='\u274c '+m;e.style.display='block'}
function show(a,role){
  const el=document.getElementById('results');
  const bc=l=>l==='Excellent'?'bg':l==='Good Start'?'bw':'bb';
  const col=s=>s>=75?'var(--good)':s>=45?'var(--warn)':'var(--bad)';
  const ring=(s,c)=>{const r=52,ci=2*Math.PI*r,d=(s/100)*ci;return `<svg width=\"120\" height=\"120\" viewBox=\"0 0 120 120\"><circle cx=\"60\" cy=\"60\" r=\"${r}\" fill=\"none\" stroke=\"rgba(255,255,255,0.06)\" stroke-width=\"10\"/><circle cx=\"60\" cy=\"60\" r=\"${r}\" fill=\"none\" stroke=\"${c}\" stroke-width=\"10\" stroke-dasharray=\"${d} ${ci}\" stroke-linecap=\"round\"/></svg>`};
  const pc=p=>{p=(p||'').toLowerCase();return p==='high'?'ph':p==='medium'?'pm':'pl'};
  const oc=col(a.overall_score),sl=a.section_labels||{},sec=a.sections||{};
  el.innerHTML=`
    <div class=\"score-hero fi\">
      <div class=\"big-score\">${ring(a.overall_score,oc)}<span class=\"num\" style=\"color:${oc}\">${a.overall_score}</span><span class=\"lbl\">/100</span></div>
      <div class=\"score-info\"><h2>Resume Score for <em>${role}</em></h2><p class=\"sum\">${a.summary||''}</p></div>
    </div>
    <div class=\"sgrid fi\">${['tone_style','content','structure','skills'].map(k=>{
      const n={tone_style:'Tone & Style',content:'Content',structure:'Structure',skills:'Skills'}[k];
      const s=sec[k]||0,t=sl[k]||'Needs Work';
      return `<div class=\"sc\"><div class=\"sch\"><span class=\"sct\">${n}</span><span class=\"badge ${bc(t)}\">${t}</span></div><div class=\"track\"><div class=\"fill\" data-s=\"${s}\" style=\"background:${col(s)}\"></div></div><div class=\"snr\"><span>Score</span><strong>${s}/100</strong></div></div>`
    }).join('')}</div>
    <div class=\"ats-card fi\"><div class=\"ats-h\"><span style=\"font-size:1.4rem\">🤖</span><div><h3>ATS Score \u2014 <span class=\"ats-big\">${a.ats_score}/100</span></h3><p style=\"color:var(--muted);font-size:.85rem\">How well your resume passes automated hiring filters</p></div></div><ul class=\"ats-list\">${(a.weak_points||[]).slice(0,3).map(w=>`<li>${w}</li>`).join('')}</ul></div>
    <div class=\"two fi\">
      <div class=\"panel\"><h3>\u2705 Strong Points</h3><ul>${(a.strong_points||[]).map(p=>`<li><span class=\"dot dg\"></span>${p}</li>`).join('')}</ul></div>
      <div class=\"panel\"><h3>\u26a0\ufe0f Weak Points</h3><ul>${(a.weak_points||[]).map(p=>`<li><span class=\"dot dr\"></span>${p}</li>`).join('')}</ul></div>
    </div>
    <div class=\"kw-card fi\"><h3>🔑 Missing Keywords for \"${role}\"</h3><div class=\"chips\">${(a.missing_keywords||[]).map(k=>`<span class=\"chip\">${k}</span>`).join('')}</div></div>
    <div class=\"rec-card fi\"><h3>💡 Recommendations</h3>${(a.recommendations||[]).map(r=>`<div class=\"rec\"><span class=\"pri ${pc(r.priority)}\">${r.priority}</span><span class=\"rt\">${r.action}</span></div>`).join('')}</div>
    <button class=\"again\" onclick=\"reset()\">\u21a9 Analyze Another Resume</button>`;
  el.style.display='block';
  setTimeout(()=>document.querySelectorAll('.fill').forEach(b=>b.style.width=b.dataset.s+'%'),100);
  el.scrollIntoView({behavior:'smooth',block:'start'});
}
function reset(){document.getElementById('results').style.display='none';document.getElementById('upload-section').scrollIntoView({behavior:'smooth'});document.getElementById('abtn').disabled=false;document.getElementById('err').style.display='none'}
</script>
</body>
</html>"""

with open('/content/resume_analyzer/templates/index.html', 'w', encoding='utf-8') as f:
    f.write(html)
print('index.html created!')

index.html created!


In [7]:
import os
os.environ['GROQ_API_KEY'] = 'write groq api key'   # get free key at console.groq.com/keys
print('API key set!')

API key set!


In [8]:
import os, threading, subprocess, time
from pyngrok import ngrok

os.system("pkill -f 'app.py'")
ngrok.kill()
time.sleep(2)

os.chdir('/content/resume_analyzer')
ngrok.set_auth_token('write ngrok api key')   # get free token at dashboard.ngrok.com

threading.Thread(target=lambda: subprocess.run(['python','app.py']), daemon=True).start()
time.sleep(3)

url = ngrok.connect(5000)
print(f'App is live at: {url}')

App is live at: NgrokTunnel: "https://curvy-nibble-unzip.ngrok-free.dev" -> "http://localhost:5000"
